# dCR Compression Study — Phase 7-RUN: Prune Model A, fine-tune B / B′ / C

**Read `PROJECT_MASTER.md` §§9, 10, 17 and `docs/kaggle_run_contract.md` before running this.**

Prunes Model A's trained checkpoint ONCE per ratio (0.5, 0.7 — master §9), saves the shared
kept-channel mask + its hash, then fine-tunes B (Dice only), B′ (generic boundary) and C (dCR)
at both ratios × 3 seeds each = **18 runs total**, all from that ONE shared pruned base.

This notebook is **authored only** in this phase — it is not executed here (no real Model A
checkpoint exists locally yet; Phase 6-RUN must complete first). It incorporates every lesson
from today's Phase 6-RUN Kaggle session:

  * **GPU must be T4, not P100** — Cell 1 asserts this and fails loudly otherwise.
  * **Writable repo copy** via `shutil.copytree()`, never a symlink into `/kaggle/input/` (which
    is read-only — editing `configs/*.yaml` in place there silently no-ops, no error).
  * **`batch_size=1`** — `4` OOMs; `2` was tested and confirmed *slower* than `1`
    (168.7 vs 132.1 min/run) — do not use either.
  * **`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`** — reduces fragmentation-driven OOMs
    across many sequential runs.
  * **Staged dependency install** (numpy alone → verify → scipy alone → verify → everything
    else) — installing everything in one `pip install` intermittently produced
    `numpy.dtype`/`numpy.rec`/`numpy.strings` binary-incompatibility errors.
  * **Absolute paths + explicit `cwd=`** on every subprocess call — relative paths previously
    made all 6 Phase 6-RUN runs fail instantly with `FileNotFoundError`.
  * **Push results via `kaggle datasets create`/`version`** after every completed run, using
    Kaggle Secrets for auth — `FileLink` / the Output-pane browser download are both proven
    unreliable for files this large.

Every hyperparameter is read from `configs/*.yaml` inside the writable repo copy — nothing is
restated in a notebook cell (`docs/kaggle_run_contract.md` §2).


In [ ]:
# CELL 1 -- Environment check: GPU must be T4, not P100 (Phase 7-RUN lesson).
import sys

print("Python:", sys.version)

import torch
print("torch:", torch.__version__, "| CUDA:", torch.version.cuda,
      "| available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU visible. Set this notebook's Accelerator to GPU (T4 x2 or T4 x1) in the "
        "settings panel, then Restart & Run All.")

gpu_name = torch.cuda.get_device_name(0)
print("GPU:", gpu_name)

# Phase 7-RUN lesson: this session specifically needed a T4 (not P100 -- P100 was unavailable/
# behaved differently under this notebook's memory profile). Fail loudly here rather than
# silently training on the wrong accelerator and only noticing later from run duration.
if "T4" not in gpu_name:
    raise RuntimeError(
        f"Expected a T4 GPU for this notebook (Phase 7-RUN), got {gpu_name!r}. "
        f"Select 'GPU T4 x2' (or T4 x1) in the notebook's Accelerator settings -- not P100 -- "
        f"then Restart & Run All.")

print("\nGPU check PASSED: T4 confirmed.")


In [ ]:
# CELL 2 -- Writable repo copy (never symlink into /kaggle/input/, which is read-only),
# batch_size=1, PYTORCH_CUDA_ALLOC_CONF. All Phase 7-RUN lessons.
import os
import shutil
from pathlib import Path

# <<< EDIT THIS if your dataset slug differs from what you named it under "Add Data" >>>
REPO_INPUT = Path("/kaggle/input/dcr-compression-study-repo")
REPO_ROOT = Path("/kaggle/working/dcr-compression-study-repo")

assert REPO_INPUT.exists(), (
    f"repo dataset not found at {REPO_INPUT} -- check Add Data > Your Datasets, "
    f"the slug must match this path exactly")

# Phase 7-RUN lesson: /kaggle/input/ is READ-ONLY. Editing configs/*.yaml in place (the
# batch_size edit below) on a path under /kaggle/input/ silently no-ops -- no error is raised,
# the edit just does not persist, and training proceeds with the UNEDITED value. NEVER symlink
# into /kaggle/input/; copy the whole repo into /kaggle/working/ first, which IS writable.
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
print(f"copying {REPO_INPUT} -> {REPO_ROOT} (writable copy, this can take a minute)...")
shutil.copytree(REPO_INPUT, REPO_ROOT)
print("copy complete.")

# Phase 7-RUN lesson: batch_size=4 (training_shared.yaml's default) OOMs pruning/fine-tuning a
# full U-Net at 1024x1024 on a T4. batch_size=2 was ALSO tested and confirmed SLOWER than
# batch_size=1 (168.7 vs 132.1 min/run). Set batch_size=1 HERE, in the writable copy,
# immediately after copying -- before anything else touches these configs.
import yaml

shared_cfg_path = REPO_ROOT / "configs" / "training_shared.yaml"
shared_cfg = yaml.safe_load(shared_cfg_path.read_text(encoding="utf-8"))
shared_cfg.setdefault("training", {})["batch_size"] = 1
shared_cfg_path.write_text(yaml.safe_dump(shared_cfg, sort_keys=False), encoding="utf-8")
resolved_bs = yaml.safe_load(shared_cfg_path.read_text(encoding="utf-8"))["training"]["batch_size"]
print(f"training_shared.yaml: batch_size set to {resolved_bs} "
      f"(Phase 7-RUN lesson: 4 OOMs, 2 is SLOWER than 1 -- 168.7 vs 132.1 min/run)")

# Phase 7-RUN lesson: without this, repeated alloc/free cycles across 18 runs fragment CUDA's
# caching allocator and can OOM even when total usage is well under the GPU's memory.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("PYTORCH_CUDA_ALLOC_CONF =", os.environ["PYTORCH_CUDA_ALLOC_CONF"])


In [ ]:
# CELL 3 -- Staged dependency install: numpy alone + verify, THEN scipy alone + verify, THEN
# everything else in one batch. Phase 7-RUN lesson: installing everything in a single
# `pip install` (as the earlier Phase 6-RUN notebook did) intermittently produced
# numpy.dtype/numpy.rec/numpy.strings binary-incompatibility errors -- pip's dependency
# resolver does not guarantee install ORDER, and a numpy ABI mismatch between a package's
# compiled extension and whatever numpy build lands last is silent until first import. This
# exact staged order (numpy alone -> scipy alone -> everything else) has been confirmed to
# avoid it.
import subprocess
import sys


def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


print("stage 1/3: numpy alone")
pip_install("numpy==1.24.3")
import numpy
print("  numpy", numpy.__version__, "imports OK")

print("stage 2/3: scipy alone")
pip_install("scipy==1.11.3")
import scipy
print("  scipy", scipy.__version__, "imports OK")

print("stage 3/3: everything else, one batch")
# A34: torch/torchvision are NOT reinstalled -- Kaggle's preinstalled, GPU-matched build is
# used as-is (reinstalling risks silently breaking CUDA).
OTHER_DEPS = [
    "opencv-python==4.8.1.78", "scikit-image==0.21.0",
    "tifffile==2023.7.10", "pyyaml==6.0.1", "pandas==2.1.1",
    "statsmodels==0.14.0", "imagehash==4.3.1", "thop==0.1.1.post2209072238",
]
pip_install(*OTHER_DEPS)
print("  installed:", ", ".join(OTHER_DEPS))

print("\ndependencies installed (torch/torchvision left as Kaggle's own build -- A34)")


In [ ]:
# CELL 4 -- Full import verification of every dependency, before spending any GPU time.
import importlib

REQUIRED_MODULES = [
    "torch", "numpy", "scipy", "cv2", "skimage", "tifffile", "yaml", "pandas",
    "statsmodels", "imagehash", "thop",
]
print("Verifying every dependency actually imports:\n")
failed = []
for mod_name in REQUIRED_MODULES:
    try:
        mod = importlib.import_module(mod_name)
        version = getattr(mod, "__version__", "?")
        print(f"  OK   {mod_name:<14} {version}")
    except Exception as e:
        print(f"  FAIL {mod_name:<14} {type(e).__name__}: {e}")
        failed.append(mod_name)

assert not failed, (
    f"failed to import: {failed} -- fix before proceeding (see Cell 3's staged install order)")
print("\nAll dependencies import cleanly.")


In [ ]:
# CELL 5 -- Repo/data sanity check: assert the writable path and Model A's checkpoint exist,
# then os.chdir() into the repo root.
import os
import sys
from pathlib import Path

required = [
    "src", "configs", "experiments",
    "data/processed/preprocess_manifest.csv",
    "data/splits/train.csv", "data/splits/test.csv",
    "data/curvature/curvature_bins.csv",
    "requirements-kaggle.txt",
]
print(f"Checking the writable repo copy at {REPO_ROOT} contains everything training needs:\n")
missing = []
for rel in required:
    p = REPO_ROOT / rel
    ok = p.exists()
    print(f"  {'OK ' if ok else 'MISSING'}  {rel}")
    if not ok:
        missing.append(rel)
assert not missing, f"missing paths in the writable repo copy: {missing}"

# <<< EDIT THIS once Model A has actually been trained (Phase 6-RUN) and its checkpoint
#     downloaded back into checkpoints/A_dcr_seed1/best.pt at the repo root (or included in
#     the uploaded repo dataset) >>>
MODEL_A_CHECKPOINT = REPO_ROOT / "checkpoints" / "A_dcr_seed1" / "best.pt"
assert MODEL_A_CHECKPOINT.exists(), (
    f"Model A's checkpoint not found at {MODEL_A_CHECKPOINT}. This notebook prunes and "
    f"fine-tunes B/B'/C FROM Model A (master §9) -- Phase 6-RUN must complete and "
    f"checkpoints/A_dcr_seed1/ must be part of the repo dataset (or copied into this writable "
    f"copy) before this notebook can run.")
print(f"\nModel A checkpoint found: {MODEL_A_CHECKPOINT}")

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("cwd ->", Path.cwd())


In [ ]:
# CELL 6 -- Prune Model A's checkpoint ONCE per ratio (master §9), save+hash the masks, then
# a BN-recalibration DRY RUN at the primary ratio -- proving the full prune -> recalibrate
# pipeline (that each of B/B'/C's own single-job run in Cell 8 will independently redo) works,
# before spending any fine-tuning GPU time on it. Prints mask hash + param count.
from types import SimpleNamespace

from src.compression.mask_io import load_channel_mask  # noqa: F401 (re-exported for Cell 8 use)
from src.compression.structured_pruning import StructuredPruner
from src.config import compose_run_config, load_yaml
from src.datasets.dataset import build_dataloaders
from src.models.unet import UNet
from src.training.checkpointing import checkpoint_hash, load_checkpoint
from src.training.seed_manager import seeds_for_run

# The real Kaggle runner path added this phase (A42) -- Cell 8 calls this same code indirectly
# via experiments/multi_seed_train.py's main(); this cell calls it directly for the dry run.
from experiments.multi_seed_train import _build_pruned_base

pruning_cfg = load_yaml("pruning.yaml")["pruning"]
RATIOS = pruning_cfg["ratios"]  # frozen [0.5, 0.7], master §9
MIN_KEEP = int(pruning_cfg["min_channels_per_layer"])
model_a_cfg = load_yaml("model_a_dcr.yaml")["model"]

print(f"Model A checkpoint: {MODEL_A_CHECKPOINT}")
print(f"  sha256: {checkpoint_hash(MODEL_A_CHECKPOINT)}")

pruner = StructuredPruner()
MASK_DIR = REPO_ROOT / "checkpoints" / "channel_masks"
MASK_DIR.mkdir(parents=True, exist_ok=True)

mask_hashes = {}
for ratio in RATIOS:
    print(f"\n{'=' * 74}\nratio {ratio}\n{'=' * 74}")
    fresh = UNet.from_config({"model": model_a_cfg})
    load_checkpoint(MODEL_A_CHECKPOINT, model=fresh, map_location="cpu")

    mask_path = MASK_DIR / f"ratio_{ratio}.json"
    if mask_path.exists():
        print(f"  mask already exists at {mask_path} -- LOADING, not recomputing "
              f"(master §9: prune once per ratio)")
    pruned_model, record = pruner.prune(fresh, ratio=ratio, mask_path=str(mask_path),
                                        min_keep=MIN_KEEP)
    mask_hashes[ratio] = record["mask_hash"]
    print(f"  mask_hash: {record['mask_hash']}")
    print(f"  params: {record['params_before']:,} -> {record['params_after']:,} "
          f"({100 * (1 - record['params_after'] / record['params_before']):.1f}% reduction)")
    print(f"  flops:  {record['flops_before']:,} -> {record['flops_after']:,}")
    del fresh, pruned_model

# BN-recalibration dry run at the PRIMARY ratio, using the real dataloaders and the real
# _build_pruned_base() Cell 8's runs will each call independently.
primary_ratio = pruning_cfg["primary_ratio"]
b_cfg = compose_run_config("model_b_pruned.yaml")
b_cfg["pruning"]["source_checkpoint"] = str(MODEL_A_CHECKPOINT.relative_to(REPO_ROOT))
b_cfg["pruning"]["channel_mask"] = str(
    (MASK_DIR / f"ratio_{primary_ratio}.json").relative_to(REPO_ROOT))
seeds = seeds_for_run(b_cfg, 0)
train_loader, _val_loader, _train_eval_loader = build_dataloaders(
    b_cfg, seeds.augmentation_seed, seeds.data_order_seed)

args_stub = SimpleNamespace(output_root=str(REPO_ROOT), device="cuda",
                            channel_mask_hash=mask_hashes[primary_ratio])
print(f"\n{'=' * 74}\nBN-recalibration dry run at ratio {primary_ratio}\n{'=' * 74}")
_dryrun_model, dryrun_note = _build_pruned_base(b_cfg, b_cfg["pruning"], args_stub, train_loader)
print(f"  {dryrun_note}")
del _dryrun_model

print("\n\nDRY RUN before spending any fine-tuning GPU time -- confirm above:")
print("  * a real, non-trivial params/FLOPs reduction at BOTH ratios")
print("  * BN recalibration completed with a real batch count and fingerprint")
print("  * mask_hash values you are prepared to see repeated IDENTICALLY across all of B/B'/C")
print("\nMask hashes (wired into RUN_PLAN via --channel-mask-hash in Cell 8):")
for ratio, h in mask_hashes.items():
    print(f"  ratio {ratio}: {h}")


In [ ]:
# CELL 7 -- RUN_PLAN: an EDITABLE list of 18 (config, ratio, seed_index) tuples for B/B'/C
# (master §10: B/B'/C x 2 ratios x 3 seeds = 18). Structured as a plain list so a LATER session
# can trim it to only the not-yet-completed entries (see Cell 9) -- append_run's
# per-(model, ratio, seed) row means partial progress across sessions is never lost.
RUN_PLAN = [
    (cfg, ratio, seed_idx)
    for cfg in ("configs/model_b_pruned.yaml", "configs/model_b_prime_generic.yaml",
               "configs/model_c_pruned_dcr.yaml")
    for ratio in RATIOS
    for seed_idx in (0, 1, 2)
]
assert len(RUN_PLAN) == 18, f"expected 18 (B/B'/C x 2 ratios x 3 seeds), got {len(RUN_PLAN)}"

print(f"Run plan: {len(RUN_PLAN)} runs (master §10: 3 seeds per model x ratio, logged separately)\n")
for cfg, ratio, seed_idx in RUN_PLAN:
    print(f"  {cfg:<40} ratio={ratio}  seed_index={seed_idx}")

# <<< Before "Restart & Run All" in a LATER session, trim this list to exclude any
#     (model, ratio, seed) combination that already has a completed row in
#     results/tables/experiment_log.csv -- see Cell 9's printed reminder. >>>


In [ ]:
# CELL 8 -- The 18-run training loop. ABSOLUTE paths + explicit cwd=REPO_ROOT on every
# subprocess call (Phase 7-RUN lesson: relative paths made all 6 Phase 6-RUN runs fail
# INSTANTLY with FileNotFoundError). After EVERY completed run (returncode == 0 only), push
# results via `kaggle datasets create`/`version` using Kaggle Secrets for auth, then print the
# exact local `kaggle datasets download` command -- NOT FileLink or the Output pane's browser
# download, both proven unreliable for files this large.
import json
import shutil
import subprocess
import sys
import time

from kaggle_secrets import UserSecretsClient

LOG_DIR = REPO_ROOT / "_notebook_run_logs"
LOG_DIR.mkdir(exist_ok=True)

secrets = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")

# <<< EDIT if you name the results dataset differently >>>
RESULTS_DATASET_SLUG = "dcr-phase7-bc-results"
RESULTS_DATASET_DIR = REPO_ROOT / "_kaggle_dataset_push"
RESULTS_DATASET_DIR.mkdir(exist_ok=True)


def push_results_to_kaggle_dataset(tag: str) -> None:
    """Copy this run's checkpoints/logs/experiment_log.csv into the push staging dir and
    create/version the private Kaggle Dataset (Phase 7-RUN lesson: FileLink is unreliable)."""
    for sub in ("checkpoints", "logs"):
        src = REPO_ROOT / sub
        if src.exists():
            shutil.copytree(src, RESULTS_DATASET_DIR / sub, dirs_exist_ok=True)
    tables_src = REPO_ROOT / "results" / "tables" / "experiment_log.csv"
    if tables_src.exists():
        dst = RESULTS_DATASET_DIR / "results" / "tables"
        dst.mkdir(parents=True, exist_ok=True)
        shutil.copy(tables_src, dst / "experiment_log.csv")

    meta_path = RESULTS_DATASET_DIR / "dataset-metadata.json"
    is_new = not meta_path.exists()
    meta_path.write_text(json.dumps({
        "title": RESULTS_DATASET_SLUG,
        "id": f"{os.environ['KAGGLE_USERNAME']}/{RESULTS_DATASET_SLUG}",
        "licenses": [{"name": "CC0-1.0"}],
    }, indent=2), encoding="utf-8")

    cmd = (["kaggle", "datasets", "create", "-p", str(RESULTS_DATASET_DIR), "-r", "zip"]
          if is_new else
          ["kaggle", "datasets", "version", "-p", str(RESULTS_DATASET_DIR), "-r", "zip",
           "-m", f"after {tag}"])
    proc = subprocess.run(cmd, cwd=str(REPO_ROOT), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True)
    print(f"  kaggle datasets {'create' if is_new else 'version'} -> "
          f"{'OK' if proc.returncode == 0 else 'FAILED'}")
    print("  " + "\n  ".join(proc.stdout.splitlines()[-8:]))
    print(f"  Download locally with:\n"
          f"    kaggle datasets download -d {os.environ['KAGGLE_USERNAME']}/"
          f"{RESULTS_DATASET_SLUG} --unzip -p ./kaggle_downloads")


results = []
for cfg, ratio, seed_idx in RUN_PLAN:
    model_name = Path(cfg).stem.replace("model_", "")
    tag = f"{model_name}_ratio{ratio}_seedidx{seed_idx}"
    print(f"\n{'=' * 74}\nSTARTING {tag}\n{'=' * 74}")
    t0 = time.time()

    script = REPO_ROOT / "experiments" / "multi_seed_train.py"
    cmd = [sys.executable, str(script),
          "--model-config", cfg, "--seed-index", str(seed_idx),
          "--device", "cuda", "--output-root", str(REPO_ROOT),
          "--channel-mask-hash", mask_hashes[ratio],
          "--executed-on", "kaggle"]
    proc = subprocess.run(cmd, cwd=str(REPO_ROOT),
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    log_path = LOG_DIR / f"{tag}.log"
    log_path.write_text(proc.stdout, encoding="utf-8")
    print("\n".join(proc.stdout.splitlines()[-25:]))  # tail, so problems are visible inline

    dur_min = (time.time() - t0) / 60
    ok = proc.returncode == 0
    print(f"\n--- {tag}: {'OK' if ok else 'FAILED -- see ' + str(log_path)} "
          f"({dur_min:.1f} min) ---")
    results.append({"tag": tag, "config": cfg, "ratio": ratio, "seed_index": seed_idx,
                    "returncode": proc.returncode, "duration_min": round(dur_min, 1),
                    "log": str(log_path)})

    if ok:
        push_results_to_kaggle_dataset(tag)
    else:
        print(f"  *** {tag} FAILED -- NOT pushing (would bury good results under a failed "
              f"run's partial state). Fix and re-run this entry before continuing RUN_PLAN. ***")

print("\n\nSUMMARY")
for r in results:
    print(f"  {'OK ' if r['returncode'] == 0 else 'ERR'}  {r['tag']:<40} "
          f"{r['duration_min']:>6.1f} min")
n_failed = sum(1 for r in results if r["returncode"] != 0)
if n_failed:
    print(f"\n*** {n_failed}/{len(results)} run(s) FAILED. Check their .log files under "
          f"{LOG_DIR} before trusting the pushed dataset. ***")


In [ ]:
# CELL 9 -- Summary + next-steps reminder to update RUN_PLAN for future sessions.
import pandas as pd

log_path = REPO_ROOT / "results" / "tables" / "experiment_log.csv"
if log_path.exists():
    df = pd.read_csv(log_path)
    print(f"experiment_log.csv now has {len(df)} row(s) total.\n")
else:
    df = None
    print("No experiment_log.csv found yet.\n")

completed_this_session = [r["tag"] for r in results if r["returncode"] == 0]
print(f"Completed this session: {len(completed_this_session)}/{len(RUN_PLAN)}")
for tag in completed_this_session:
    print(f"  OK  {tag}")

failed_this_session = [r["tag"] for r in results if r["returncode"] != 0]
if failed_this_session:
    print(f"\nFAILED this session ({len(failed_this_session)}) -- re-run these specific "
          f"entries first in a future session:")
    for tag in failed_this_session:
        print(f"  FAIL  {tag}")

print(f"\n{'=' * 74}")
print("NEXT STEPS FOR A FUTURE SESSION")
print(f"{'=' * 74}")
print("1. Download this session's results locally:")
print(f"   kaggle datasets download -d $KAGGLE_USERNAME/{RESULTS_DATASET_SLUG} "
      f"--unzip -p ./kaggle_downloads")
print("2. Merge kaggle_downloads/results/tables/experiment_log.csv rows into the repo's own")
print("   copy (do not overwrite -- the header must stay identical), and place checkpoints/ +")
print("   logs/ at the repo root (both gitignored, always safe to place).")
print("3. Before a new Kaggle session's Restart & Run All, EDIT Cell 7's RUN_PLAN to remove")
print("   any (model, ratio, seed) combination that already has a completed row in")
print("   results/tables/experiment_log.csv -- otherwise completed work is repeated (harmless")
print("   but wastes GPU quota).")
print("4. Once all 18 rows are complete, author and run a Phase 7-RUN verification script")
print("   (mirroring audit/verify_phase6_run.py's pattern) confirming: all 18 checkpoints")
print("   load; B/B'/C at each ratio report IDENTICAL measured params/FLOPs and an identical")
print("   mask_hash (master §9); and no two of Vanilla/A/B/B'/C collapse to identical outputs")
print("   on a probe image (Golden Rule 2).")
